# 🧠 AutoAtlas Medical Image Pipeline
### Cross-Domain MRI ↔ CT Translation & Unsupervised Segmentation

This notebook consolidates the project's logic into a single executive environment for demonstration and research purposes.

## 🛠 Setup & Dependencies

In [ ]:
import os
import io
import cv2
import numpy as np
import nibabel as nib
import SimpleITK as sitk
import sqlite3
import uuid
import time
from datetime import datetime
from PIL import Image
from dotenv import load_dotenv

load_dotenv()

DB_PATH = "autoatlas.db"
os.makedirs("data/uploads", exist_ok=True)
os.makedirs("data/results", exist_ok=True)

## 📂 Utility Functions

In [ ]:
def ensure_dirs():
    os.makedirs("data/uploads", exist_ok=True)
    os.makedirs("data/results", exist_ok=True)

def detect_modality(file_path):
    filename = os.path.basename(file_path).lower()
    if 'mri' in filename or 't1' in filename or 't2' in filename:
        return 'MRI'
    if 'ct' in filename:
        return 'CT'
    return 'MRI'

def load_image_as_rgb(file_path):
    ext = file_path.lower()
    if ext.endswith(('.png', '.jpg', '.jpeg')):
        img = Image.open(file_path).convert('RGB')
        return np.array(img)
    elif ext.endswith(('.nii', '.nii.gz')):
        try:
            img = nib.load(file_path)
            data = img.get_fdata()
            slice_2d = data[:, :, data.shape[2]//2]
            slice_2d = (slice_2d - np.min(slice_2d)) / (np.max(slice_2d) - np.min(slice_2d) + 1e-8) * 255.0
            slice_rgb = cv2.cvtColor(slice_2d.astype(np.uint8), cv2.COLOR_GRAY2RGB)
            return slice_rgb
        except:
            return np.zeros((256, 256, 3), dtype=np.uint8)
    return np.zeros((256, 256, 3), dtype=np.uint8)

## 🗄 Database Management

In [ ]:
def init_db():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS sessions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            session_id TEXT UNIQUE,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS uploads (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            session_id TEXT,
            filename TEXT,
            modality TEXT,
            file_path TEXT,
            uploaded_at DATETIME DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (session_id) REFERENCES sessions(session_id)
        )
    ''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS results (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            upload_id INTEGER,
            translated_path TEXT,
            segmented_path TEXT,
            atlas_used TEXT,
            processed_at DATETIME DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (upload_id) REFERENCES uploads(id)
        )
    ''')
    conn.commit()
    conn.close()

def create_session():
    session_id = str(uuid.uuid4())
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('INSERT INTO sessions (session_id) VALUES (?)', (session_id,))
    conn.commit()
    conn.close()
    return session_id

def save_upload(session_id, filename, modality, file_path):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        INSERT INTO uploads (session_id, filename, modality, file_path) 
        VALUES (?, ?, ?, ?)
    ''', (session_id, filename, modality, file_path))
    upload_id = cursor.lastrowid
    conn.commit()
    conn.close()
    return upload_id

def save_result(upload_id, translated_path, segmented_path, atlas_used):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''
        INSERT INTO results (upload_id, translated_path, segmented_path, atlas_used) 
        VALUES (?, ?, ?, ?)
    ''', (upload_id, translated_path, segmented_path, atlas_used))
    conn.commit()
    conn.close()

init_db()

## 🧠 Translation & Segmentation Models

In [ ]:
def mock_cyclegan_translation(input_path, output_dir, source_modality):
    img = load_image_as_rgb(input_path)
    if source_modality == 'MRI':
        translated = cv2.applyColorMap(cv2.equalizeHist(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)), cv2.COLORMAP_BONE)
    else:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        cl1 = clahe.apply(gray)
        translated = cv2.cvtColor(cl1, cv2.COLOR_GRAY2RGB)
    
    filename = os.path.basename(input_path)
    target_modality = 'CT' if source_modality == 'MRI' else 'MRI'
    out_name = f"translated_{target_modality}_{filename}.png"
    out_path = os.path.join(output_dir, out_name)
    Image.fromarray(translated).save(out_path)
    return out_path

def mock_autoatlas_segmentation(input_path, output_dir, atlas_type="MNI152"):
    img = load_image_as_rgb(input_path)
    mask = np.zeros_like(img)
    h, w = img.shape[:2]
    if atlas_type == "MNI152":
        cv2.ellipse(mask, (w//2, h//2), (w//4, int(h/2.5)), 0, 0, 360, (255, 0, 0), -1)
        cv2.circle(mask, (w//2 + 20, h//2 - 20), 20, (0, 0, 255), -1)
    else:
        cv2.circle(mask, (w//3, h//2), 40, (0, 255, 0), -1)
        cv2.circle(mask, (2*w//3, h//2), 30, (255, 0, 255), -1)
    
    alpha = 0.5
    segmented = cv2.addWeighted(img, 1-alpha, mask, alpha, 0)
    filename = os.path.basename(input_path)
    out_name = f"segmented_{atlas_type}_{filename}.png"
    out_path = os.path.join(output_dir, out_name)
    Image.fromarray(segmented).save(out_path)
    return out_path

## 📝 Diagnostic Report Generation

In [ ]:
client = OpenAI(
    api_key=os.getenv("MISTRAL_API_KEY"),
    base_url="https://api.mistral.ai/v1"
)

def generate_diagnostic_report(modality, atlas_type, model_name="open-mistral-7b"):
    prompt = f"Medical Imaging Diagnostic Report\nModality: {modality}\nAtlas Registered: {atlas_type}\nFindings: Based on the structural analysis, we observe "
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are a professional radiologist providing concise diagnostic summaries based on automated image processing results."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=150,
            temperature=0.7
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error generating report via Mistral: {str(e)}"

## 🚀 Pipeline Execution

In [ ]:
def run_pipeline(input_file_path, atlas_choice="MNI152", enable_translation=True):
    session_id = create_session()
    modality = detect_modality(input_file_path)
    print(f"Processing Session: {session_id}")
    print(f"Detected Modality: {modality}")
    
    upload_id = save_upload(session_id, os.path.basename(input_file_path), modality, input_file_path)
    session_res_dir = os.path.join("data/results", session_id)
    os.makedirs(session_res_dir, exist_ok=True)
    
    # 1. Translation
    translated_path = None
    if enable_translation:
        target = "CT" if modality == "MRI" else "MRI"
        print(f"Generating Synthetic {target}...")
        translated_path = mock_cyclegan_translation(input_file_path, session_res_dir, modality)
    
    # 2. Segmentation
    print("Running AutoAtlas Segmentation...")
    input_for_seg = translated_path if enable_translation else input_file_path
    segmented_path = mock_autoatlas_segmentation(input_for_seg, session_res_dir, atlas_choice)
    
    # 3. Report
    print("Generating AI Report...")
    report_text = generate_diagnostic_report(modality, atlas_choice)
    
    # 4. Save
    save_result(upload_id, translated_path, segmented_path, atlas_choice)
    
    return {
        "session_id": session_id,
        "segmented_path": segmented_path,
        "translated_path": translated_path,
        "report": report_text
    }

# Example usage (commented out as it requires an input file)
# result = run_pipeline("path/to/scan.png")
# print(result['report'])